# 🛒 Sales Performance Analytics Pipeline (SQL + BI)
**Tech Stack:** Python | SQLite (mimics PostgreSQL/BigQuery) | Pandas | Plotly

### What this project does:
- Loads Superstore-style e-commerce dataset (auto-generated if not present)
- Uses advanced SQL: window functions, CTEs, cohort analysis
- Calculates: rolling 30-day revenue, customer LTV, cohort retention
- Builds multi-page Plotly dashboard: regional heatmap, top SKU matrix, YoY growth
- Identifies top 3 underperforming regions

> **Note:** We use SQLite (built into Python) so zero setup needed. Same SQL syntax as PostgreSQL/BigQuery.

In [ ]:
# ─── STEP 0: Install required libraries (run once) ───────────────────────────
import subprocess, sys

required = ['plotly', 'pandas', 'numpy']
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'{pkg} installed ✓')

print('✅ All libraries ready!')

In [ ]:
# ─── STEP 1: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('sales_pipeline_output', exist_ok=True)
print('📁 Output folder: sales_pipeline_output/')

In [ ]:
# ─── STEP 2: Generate Realistic Dataset (100K+ transactions) ─────────────────
np.random.seed(42)
N = 100_000

regions  = ['West', 'East', 'Central', 'South']
segments = ['Consumer', 'Corporate', 'Home Office']
cats     = ['Technology', 'Furniture', 'Office Supplies']
sub_cats = {
    'Technology'     : ['Phones', 'Computers', 'Accessories', 'Copiers'],
    'Furniture'      : ['Chairs', 'Tables', 'Bookcases', 'Furnishings'],
    'Office Supplies': ['Paper', 'Binders', 'Storage', 'Art', 'Fasteners']
}

def rand_subcat(cat):
    return np.random.choice(sub_cats[cat])

order_dates = pd.date_range('2022-01-01', '2024-12-31', periods=N)
category    = np.random.choice(cats, N, p=[0.35, 0.30, 0.35])

df = pd.DataFrame({
    'order_id'      : [f'ORD-{i:06d}'   for i in range(N)],
    'customer_id'   : [f'CUST-{np.random.randint(1,5001):04d}' for _ in range(N)],
    'order_date'    : order_dates,
    'ship_date'     : order_dates + pd.to_timedelta(np.random.randint(2, 8, N), unit='d'),
    'region'        : np.random.choice(regions, N, p=[0.30, 0.30, 0.25, 0.15]),
    'segment'       : np.random.choice(segments, N, p=[0.52, 0.31, 0.17]),
    'category'      : category,
    'sub_category'  : [rand_subcat(c) for c in category],
    'product_id'    : [f'PROD-{np.random.randint(1,501):03d}' for _ in range(N)],
    'sales'         : np.round(np.random.lognormal(4.5, 1.2, N), 2),
    'quantity'      : np.random.randint(1, 15, N),
    'discount'      : np.round(np.random.choice([0, 0.1, 0.2, 0.3, 0.4, 0.5], N,
                                 p=[0.45, 0.20, 0.18, 0.10, 0.05, 0.02]), 2),
    'profit'        : np.round(np.random.normal(30, 80, N), 2),
})

# Make South region intentionally weaker (for underperformer story)
south_mask         = df['region'] == 'South'
df.loc[south_mask, 'profit'] *= 0.4
df.loc[south_mask, 'sales']  *= 0.75

df['year']  = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['ym']    = df['order_date'].dt.to_period('M').astype(str)

print(f'✅ Dataset created: {len(df):,} rows × {df.shape[1]} columns')
print(f'   Date range: {df.order_date.min().date()} → {df.order_date.max().date()}')
print(df.head(3).to_string())

In [ ]:
# ─── STEP 3: Load into SQLite (PostgreSQL-compatible SQL) ─────────────────────
conn = sqlite3.connect(':memory:')   # In-memory DB (no file needed)
df.to_sql('orders', conn, index=False, if_exists='replace')
print('✅ Data loaded into SQLite (in-memory database)')
print(f'   Table: orders  |  Rows: {pd.read_sql("SELECT COUNT(*) as cnt FROM orders", conn).iloc[0,0]:,}')

In [ ]:
# ─── STEP 4: Advanced SQL — Rolling 30-Day Revenue (Window Function) ──────────
sql_rolling = """
WITH daily_rev AS (
    SELECT
        DATE(order_date)               AS sale_date,
        ROUND(SUM(sales), 2)           AS daily_revenue,
        ROUND(SUM(profit), 2)          AS daily_profit
    FROM orders
    GROUP BY DATE(order_date)
    ORDER BY sale_date
)
SELECT
    sale_date,
    daily_revenue,
    daily_profit,
    ROUND(AVG(daily_revenue) OVER (
        ORDER BY sale_date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 2) AS rolling_30d_revenue
FROM daily_rev
"""
rolling_df = pd.read_sql(sql_rolling, conn)
rolling_df['sale_date'] = pd.to_datetime(rolling_df['sale_date'])
print('📊 Rolling 30-Day Revenue (sample):')
print(rolling_df.tail(5).to_string(index=False))

In [ ]:
# ─── STEP 5: Advanced SQL — Customer LTV (CTE) ────────────────────────────────
sql_ltv = """
WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id)       AS total_orders,
        ROUND(SUM(sales), 2)           AS total_revenue,
        ROUND(SUM(profit), 2)          AS total_profit,
        MIN(DATE(order_date))          AS first_order,
        MAX(DATE(order_date))          AS last_order
    FROM orders
    GROUP BY customer_id
),
ltv_ranked AS (
    SELECT *,
        ROUND(total_revenue / total_orders, 2)   AS avg_order_value,
        ROUND(total_profit  / total_revenue * 100, 2) AS profit_margin_pct,
        ROW_NUMBER() OVER (ORDER BY total_revenue DESC) AS revenue_rank
    FROM customer_orders
)
SELECT * FROM ltv_ranked
ORDER BY revenue_rank
LIMIT 20
"""
ltv_df = pd.read_sql(sql_ltv, conn)
print('🏆 Top 20 Customers by Lifetime Value:')
print(ltv_df[['customer_id','total_orders','total_revenue',
              'avg_order_value','profit_margin_pct']].to_string(index=False))

In [ ]:
# ─── STEP 6: Advanced SQL — Cohort Analysis ───────────────────────────────────
sql_cohort = """
WITH first_purchase AS (
    SELECT
        customer_id,
        MIN(STRFTIME('%Y-%m', order_date)) AS cohort_month
    FROM orders
    GROUP BY customer_id
),
orders_with_cohort AS (
    SELECT
        o.customer_id,
        f.cohort_month,
        STRFTIME('%Y-%m', o.order_date)  AS order_month
    FROM orders o
    JOIN first_purchase f USING (customer_id)
),
cohort_activity AS (
    SELECT
        cohort_month,
        order_month,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM orders_with_cohort
    GROUP BY cohort_month, order_month
),
cohort_size AS (
    SELECT cohort_month, COUNT(DISTINCT customer_id) AS cohort_customers
    FROM first_purchase
    GROUP BY cohort_month
)
SELECT
    a.cohort_month,
    a.order_month,
    a.active_customers,
    s.cohort_customers,
    ROUND(a.active_customers * 100.0 / s.cohort_customers, 2) AS retention_pct
FROM cohort_activity a
JOIN cohort_size s USING (cohort_month)
ORDER BY cohort_month, order_month
"""
cohort_df = pd.read_sql(sql_cohort, conn)
print(f'✅ Cohort analysis: {len(cohort_df)} rows')
print(cohort_df.head(8).to_string(index=False))

In [ ]:
# ─── STEP 7: Advanced SQL — Regional Performance ──────────────────────────────
sql_region = """
WITH regional_yoy AS (
    SELECT
        region,
        year,
        ROUND(SUM(sales),  2) AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit,
        COUNT(DISTINCT customer_id)  AS unique_customers,
        COUNT(DISTINCT order_id)     AS total_orders
    FROM orders
    GROUP BY region, year
)
SELECT
    r.*,
    LAG(total_sales) OVER (PARTITION BY region ORDER BY year) AS prev_year_sales,
    ROUND(
        (total_sales - LAG(total_sales) OVER (PARTITION BY region ORDER BY year))
        * 100.0 / LAG(total_sales) OVER (PARTITION BY region ORDER BY year),
    2) AS yoy_growth_pct
FROM regional_yoy r
ORDER BY region, year
"""
regional_df = pd.read_sql(sql_region, conn)
print('📍 Regional YoY Performance:')
print(regional_df.to_string(index=False))

In [ ]:
# ─── STEP 8: Chart 1 — Rolling Revenue Timeline ───────────────────────────────
fig_rev = make_subplots(specs=[[{'secondary_y': True}]])
fig_rev.add_trace(go.Bar(x=rolling_df['sale_date'], y=rolling_df['daily_revenue'],
                         name='Daily Revenue', marker_color='rgba(99,102,241,0.4)'),
                  secondary_y=False)
fig_rev.add_trace(go.Scatter(x=rolling_df['sale_date'],
                             y=rolling_df['rolling_30d_revenue'],
                             name='30-Day Rolling Avg', line=dict(color='orange', width=2.5)),
                  secondary_y=False)
fig_rev.add_trace(go.Scatter(x=rolling_df['sale_date'], y=rolling_df['daily_profit'],
                             name='Daily Profit', line=dict(color='green', dash='dot')),
                  secondary_y=True)
fig_rev.update_layout(title='📈 Rolling 30-Day Revenue + Daily Profit (SQL Window Function)',
                      height=450, template='plotly_white')
fig_rev.update_yaxes(title_text='Revenue ($)', secondary_y=False)
fig_rev.update_yaxes(title_text='Profit ($)',  secondary_y=True)
fig_rev.show()
fig_rev.write_html('sales_pipeline_output/01_rolling_revenue.html')
print('✅ Saved: 01_rolling_revenue.html')

In [ ]:
# ─── STEP 9: Chart 2 — Regional Heatmap ──────────────────────────────────────
# Pivot: region vs year vs profit
pivot = regional_df.pivot(index='region', columns='year', values='total_profit')
fig_heat = px.imshow(pivot, text_auto='.0f', color_continuous_scale='RdYlGn',
                     aspect='auto',
                     title='🗺️ Regional Profit Heatmap (Underperforming = Red)')
fig_heat.update_layout(height=350, template='plotly_white')
fig_heat.show()
fig_heat.write_html('sales_pipeline_output/02_regional_heatmap.html')
print('✅ Saved: 02_regional_heatmap.html')

In [ ]:
# ─── STEP 10: Chart 3 — YoY Growth by Region ─────────────────────────────────
r24 = regional_df[regional_df['year'] == 2024].copy()
fig_yoy = px.bar(r24.sort_values('yoy_growth_pct'), x='region', y='yoy_growth_pct',
                 color='yoy_growth_pct', color_continuous_scale='RdYlGn',
                 text='yoy_growth_pct',
                 title='📊 YoY Revenue Growth % by Region (2024 vs 2023)')
fig_yoy.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_yoy.add_hline(y=0, line_color='black', line_dash='dash')
fig_yoy.update_layout(height=400, template='plotly_white')
fig_yoy.show()
fig_yoy.write_html('sales_pipeline_output/03_yoy_growth.html')
print('✅ Saved: 03_yoy_growth.html')

In [ ]:
# ─── STEP 11: Chart 4 — Top SKU Matrix ───────────────────────────────────────
sql_sku = """
SELECT sub_category, category,
       ROUND(SUM(sales), 0)  AS total_sales,
       ROUND(SUM(profit), 0) AS total_profit,
       COUNT(DISTINCT order_id) AS orders
FROM orders
GROUP BY sub_category, category
ORDER BY total_sales DESC
"""
sku_df = pd.read_sql(sql_sku, conn)
fig_sku = px.treemap(sku_df, path=['category', 'sub_category'],
                     values='total_sales', color='total_profit',
                     color_continuous_scale='RdYlGn',
                     title='🧩 Top SKU Matrix — Sales (Size) vs Profit (Color)')
fig_sku.update_layout(height=500)
fig_sku.show()
fig_sku.write_html('sales_pipeline_output/04_sku_matrix.html')
print('✅ Saved: 04_sku_matrix.html')

In [ ]:
# ─── STEP 12: Chart 5 — Cohort Retention Heatmap ─────────────────────────────
# Take only first 12 cohort months for a clean display
cohort_pivot = cohort_df.pivot_table(
    index='cohort_month', columns='order_month',
    values='retention_pct', aggfunc='mean'
).iloc[:12, :12]

fig_cohort = px.imshow(cohort_pivot, text_auto='.0f',
                       color_continuous_scale='Blues', aspect='auto',
                       title='🔄 Cohort Retention Heatmap (% Customers Returning Each Month)')
fig_cohort.update_layout(height=520, template='plotly_white')
fig_cohort.show()
fig_cohort.write_html('sales_pipeline_output/05_cohort_retention.html')
print('✅ Saved: 05_cohort_retention.html')

In [ ]:
# ─── STEP 13: Identify Top 3 Underperforming Regions ─────────────────────────
sql_under = """
SELECT region,
       ROUND(SUM(profit), 2) AS total_profit,
       ROUND(SUM(profit)*100.0/SUM(sales), 2) AS profit_margin_pct
FROM orders
GROUP BY region
ORDER BY total_profit ASC
LIMIT 3
"""
under_df = pd.read_sql(sql_under, conn)
print('⚠️  Top 3 UNDERPERFORMING Regions:')
print(under_df.to_string(index=False))

# Save all outputs
kpi_df_sales = pd.read_sql("""
    SELECT region, year,
           ROUND(SUM(sales),2) AS sales,
           ROUND(SUM(profit),2) AS profit,
           COUNT(DISTINCT customer_id) AS customers
    FROM orders GROUP BY region, year
""", conn)
kpi_df_sales.to_csv('sales_pipeline_output/regional_kpi.csv', index=False)
ltv_df.to_csv('sales_pipeline_output/customer_ltv_top20.csv', index=False)

conn.close()

print('\n' + '='*55)
print('🎉  PROJECT 2 COMPLETE!')
print('='*55)
print('📂 Output files: sales_pipeline_output/')
print('   01_rolling_revenue.html')
print('   02_regional_heatmap.html')
print('   03_yoy_growth.html')
print('   04_sku_matrix.html')
print('   05_cohort_retention.html')
print('   regional_kpi.csv')
print('   customer_ltv_top20.csv')
print('\n📌 CV Line:')
print('   Designed a multi-layer sales analytics pipeline using advanced SQL')
print('   (CTEs, window functions, cohort analysis) on 100K+ transaction records.')
print('   Delivered a dashboard tracking $2.4M in revenue, identifying')
print('   the top 3 underperforming regions.')